# einops-einsum — ex6: scaled dot-product attention end-to-end with mask + softmax + weight heatmap

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. Running the final beacon cell reports progress against the `Einops: Deep Learning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-einsum`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.einsum — quick refresher

`einsum(*tensors, pattern)` performs sum-contraction over named indices:
1. **Elementwise** — `'i j, i j -> i j'` multiplies pointwise (no reduction).
2. **Matmul** — `'i k, k j -> i j'` contracts the shared `k` (sum-reduce).
3. **Batched** — `'b i k, b k j -> b i j'` carries `b` through, contracts `k`.
4. **Three operands** — `'i j, j k, k l -> i l'` chains two contractions; the optimizer picks pairing order.

**The two rules:**
- An index that appears on input AND output → preserved (broadcast-like).
- An index that appears on input but NOT on output → sum-contracted.

### Exercise 6 — scaled dot-product attention end-to-end with mask + softmax + weight heatmap

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose two einsum contractions (QK^T then weights@V) with masking and softmax to produce a full attention head output, and visualize the resulting attention weights.
> Keywords: attention, softmax, mask, visualization, pipeline
> ```

**KCs targeted:** `einsum-matmul-contraction`, `einsum-attention-scores`, `einsum-weighted-aggregation`

Implement `ex6_scaled_dot_product_attention(q, k, v, mask)`.

Inputs:
- `q`: `(B, T, D)` queries.
- `k`: `(B, T, D)` keys.
- `v`: `(B, T, D_v)` values.
- `mask`: `(T, T)` bool tensor — `True` means "blocked" (set score to `-inf` before softmax).

Steps:
1. **Scores** via `einsum`: `scores[b, i, j] = sum_d q[b,i,d] * k[b,j,d] / sqrt(D)`.
2. Apply `mask` (broadcast across batch) — fill blocked positions with `-inf`.
3. **Weights** via `softmax` along the key axis.
4. **Output** via a second `einsum`: `out[b, i, e] = sum_j weights[b,i,j] * v[b,j,e]`.

Return `(out, weights)` where `weights` has shape `(B, T, T)` and `out` has shape `(B, T, D_v)`. Use `einops.einsum` for **both** matmul-like steps — do not use `@`, `torch.bmm`, or `matmul`.

The test cell visualizes `weights[0]` (the first batch element's attention matrix) as a heatmap so you can see what the model is attending to under the mask.

In [ ]:
def ex6_scaled_dot_product_attention(
    q: Tensor, k: Tensor, v: Tensor, mask: Tensor
) -> tuple[Tensor, Tensor]:
    import torch.nn.functional as F
    d = q.shape[-1]
    scores = einsum(q, k, 'b i d, b j d -> b i j') / (d ** 0.5)
    scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    out = einsum(weights, v, 'b i j, b j e -> b i e')
    return out, weights


<details><summary>Solution</summary>

```python
def ex6_scaled_dot_product_attention(
    q: Tensor, k: Tensor, v: Tensor, mask: Tensor
) -> tuple[Tensor, Tensor]:
    import torch.nn.functional as F
    d = q.shape[-1]
    scores = einsum(q, k, 'b i d, b j d -> b i j') / (d ** 0.5)
    scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    out = einsum(weights, v, 'b i j, b j e -> b i e')
    return out, weights
```

**Two einsums, one pipeline.** The first call contracts `d` (the QK^T inner product). The second contracts `j` (the weighted aggregation over keys/values). Notice the second einsum's pattern `'b i j, b j e -> b i e'` looks exactly like a batched matmul — that's because it **is** a batched matmul. The point of using `einsum` here is that the index names (`i` = query position, `j` = key position, `e` = value embedding dim) make the semantics readable in a way `weights @ v` does not.

**Reading the heatmap.** With the causal mask, the upper triangle is exactly zero. Below the diagonal you should see one bright cell per row — that's the model assigning most of its weight to one key per query (a soft argmax). On random inputs the brightest cell tends to wander; on real trained attention you'd see structure (diagonals, induction-head stripes, etc.).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()